# DPO vs GRPO

Сравниваем офлайн (DPO) и онлайн (GRPO) методы alignment на Qwen2.5-0.5B.

**Метрики:**
1. **RM Reward** — средний скор Reward Model на ответах каждой модели
2. **LLM Judge** — Claude сравнивает DPO vs GRPO попарно
3. **Response Length** — средняя длина ответов (в токенах)
4. **Diversity / Entropy** — уникальность ответов

**Чекпоинты:** SFT, DPO×3 seeds, GRPO seed42

In [ ]:
import json
import os
import numpy as np
import torch
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

sft_path = "./checkpoints/sft/sft-seed42/final"
rm_path = "./checkpoints/rm/final"

dpo_seeds = [42, 43, 44]
grpo_seeds = [42]  

def get_merged_path(method, seed):
    return f"./checkpoints/{method}/{method}-seed{seed}/merged"

for method, seeds in [("dpo", dpo_seeds), ("grpo", grpo_seeds)]:
    for s in seeds:
        p = get_merged_path(method, s)
        assert os.path.isfile(os.path.join(p, "model.safetensors")), f"Missing: {p}"
        print(f"{method}-seed{s}: OK")

assert os.path.isfile(os.path.join(sft_path, "model.safetensors")), "Missing SFT"
print(f"sft-seed42: OK")
print(f"\nRM: {rm_path}")

/home/artmak/Desktop/Education/mini-LLM/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dpo-seed42: OK
dpo-seed43: OK
dpo-seed44: OK
grpo-seed42: OK
sft-seed42: OK

RM: ./checkpoints/rm/final


## 1. Генерация на 200 eval prompts

Прогоняем каждый merged checkpoint на полном наборе 200 eval промптов.


In [2]:
with open("eval_prompts.json") as f:
    eval_prompts = json.load(f)
print(f"{len(eval_prompts)} eval prompts")

def generate_all(model_path, output_path, label):
    if os.path.exists(output_path):
        with open(output_path) as f:
            existing = json.load(f)
        print(f"{label}: already have {len(existing)} generations, skipping")
        return existing

    print(f"{label}: generating on {len(eval_prompts)} prompts...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

    model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.bfloat16).cuda()
    model.resize_token_embeddings(len(tokenizer))
    model.eval()

    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    eos_ids = [tokenizer.eos_token_id, im_end_id]
    results = []

    for i, messages in enumerate(eval_prompts):
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.7,
                do_sample=True,
                eos_token_id=eos_ids,
                pad_token_id=tokenizer.pad_token_id,
            )
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        results.append({"prompt": messages[0]["content"], "response": response})
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(eval_prompts)}")

    with open(output_path, "w") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"  Saved {len(results)} generations to {output_path}")

    del model
    torch.cuda.empty_cache()
    return results

200 eval prompts


In [3]:
all_generations = {}

# SFT baseline
sft_gen_path = "./checkpoints/sft/sft-seed42/generations_200.json"
all_generations["sft-seed42"] = generate_all(sft_path, sft_gen_path, "sft-seed42")

# DPO
for seed in dpo_seeds:
    key = f"dpo-seed{seed}"
    gen_path = f"./checkpoints/dpo/{key}/generations_200.json"
    all_generations[key] = generate_all(get_merged_path("dpo", seed), gen_path, key)

# GRPO
for seed in grpo_seeds:
    key = f"grpo-seed{seed}"
    gen_path = f"./checkpoints/grpo/{key}/generations_200.json"
    all_generations[key] = generate_all(get_merged_path("grpo", seed), gen_path, key)

print(f"\nLoaded generations: {list(all_generations.keys())}")

sft-seed42: generating on 200 prompts...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 29683.68it/s]


  50/200
  100/200
  150/200
  200/200
  Saved 200 generations to ./checkpoints/sft/sft-seed42/generations_200.json
dpo-seed42: generating on 200 prompts...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 26787.68it/s]


  50/200
  100/200
  150/200
  200/200
  Saved 200 generations to ./checkpoints/dpo/dpo-seed42/generations_200.json
dpo-seed43: generating on 200 prompts...


...

  50/200
  100/200
  150/200
  200/200
  Saved 200 generations to ./checkpoints/dpo/dpo-seed43/generations_200.json
dpo-seed44: generating on 200 prompts...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 27579.09it/s]


  50/200
  100/200
  150/200
  200/200
  Saved 200 generations to ./checkpoints/dpo/dpo-seed44/generations_200.json
grpo-seed42: generating on 200 prompts...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 27273.60it/s]


  50/200
  100/200
  150/200
  200/200
  Saved 200 generations to ./checkpoints/grpo/grpo-seed42/generations_200.json

Loaded generations: ['sft-seed42', 'dpo-seed42', 'dpo-seed43', 'dpo-seed44', 'grpo-seed42']


## 2. RM Reward Scoring

Прогоняем все ответы через Reward Model, получаем средний скор по 200 промптам.

In [4]:
rm_scores_path = "./comparison_rm_scores.json"

if os.path.exists(rm_scores_path):
    with open(rm_scores_path) as f:
        rm_scores = json.load(f)
    print("Loaded cached RM scores")
else:
    print("Loading RM...")
    rm_model = AutoModelForSequenceClassification.from_pretrained(
        rm_path, num_labels=1, dtype=torch.bfloat16
    ).cuda().eval()
    rm_tokenizer = AutoTokenizer.from_pretrained(rm_path)

    rm_scores = {}

    for model_key, gens in all_generations.items():
        scores = []
        for g in gens:
            messages = [
                {"role": "user", "content": g["prompt"]},
                {"role": "assistant", "content": g["response"]},
            ]
            text = rm_tokenizer.apply_chat_template(messages, tokenize=False)
            inputs = rm_tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(rm_model.device)
            with torch.no_grad():
                score = rm_model(**inputs).logits.squeeze().item()
            scores.append(score)
        rm_scores[model_key] = scores
        print(f"{model_key}: mean={np.mean(scores):.4f}, std={np.std(scores):.4f}")

    with open(rm_scores_path, "w") as f:
        json.dump(rm_scores, f, indent=2)

    del rm_model
    torch.cuda.empty_cache()

# Сводка
print("\n=== RM Reward Summary ===")
for key, scores in rm_scores.items():
    print(f"{key:20s}  mean={np.mean(scores):+.4f}  std={np.std(scores):.4f}")

Loading RM...


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 27491.55it/s]


sft-seed42: mean=0.4174, std=0.7669
dpo-seed42: mean=0.5383, std=0.8302
dpo-seed43: mean=0.5696, std=0.8224
dpo-seed44: mean=0.5131, std=0.8215
grpo-seed42: mean=0.4815, std=0.8550

=== RM Reward Summary ===
sft-seed42            mean=+0.4174  std=0.7669
dpo-seed42            mean=+0.5383  std=0.8302
dpo-seed43            mean=+0.5696  std=0.8224
dpo-seed44            mean=+0.5131  std=0.8215
grpo-seed42           mean=+0.4815  std=0.8550


## 3. Response Length and Diversity

1. **Длина** — среднее количество слов. Reward hacking часто проявляется как удлинение ответов.
2. **Unique trigrams ratio** — доля уникальных триграмм от общего числа. 

In [5]:
def response_stats(generations):
    lengths = []
    all_trigrams = []
    total_trigrams = 0
    for g in generations:
        words = g["response"].split()
        lengths.append(len(words))
        trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
        all_trigrams.extend(trigrams)
        total_trigrams += len(trigrams)
    unique_ratio = len(set(all_trigrams)) / max(total_trigrams, 1)
    return {
        "mean_length": np.mean(lengths),
        "std_length": np.std(lengths),
        "median_length": np.median(lengths),
        "unique_trigram_ratio": unique_ratio,
    }

print(f"{'Model':20s} {'Mean len':>10s} {'Std len':>10s} {'Median':>10s} {'Trigram uniq':>12s}")
print("-" * 65)

length_stats = {}
for key, gens in all_generations.items():
    stats = response_stats(gens)
    length_stats[key] = stats
    print(f"{key:20s} {stats['mean_length']:10.1f} {stats['std_length']:10.1f} "
          f"{stats['median_length']:10.1f} {stats['unique_trigram_ratio']:12.4f}")

Model                  Mean len    Std len     Median Trigram uniq
-----------------------------------------------------------------
sft-seed42                 52.0       65.9       23.0       0.8964
dpo-seed42                 76.7       74.4       62.0       0.8801
dpo-seed43                 77.4       72.8       62.5       0.8802
dpo-seed44                 78.5       74.2       62.5       0.8739
grpo-seed42                78.4       79.8       46.0       0.8839


## 4. LLM Judge
Попарное сравнение: Claude получает prompt + два ответа (A/B), выбирает лучший.
Forward pass (DPO=A) + reverse pass (DPO=B) для контроля position bias.

Сравниваем DPO-seed42 vs GRPO-seed42 на 200 промптах.

In [12]:
import anthropic
from dotenv import load_dotenv
load_dotenv()

client = anthropic.Anthropic(api_key=os.environ["API_KEY"])

JUDGE_PROMPT = """You are evaluating two AI assistant responses to a user question.

User question: {prompt}

Response A:
{response_a}

Response B:
{response_b}

Which response is better? Consider helpfulness, accuracy, and clarity.
Reply with ONLY one of: A, B, or TIE"""

def judge_pair(prompt, resp_a, resp_b):
    msg = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=16,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(
            prompt=prompt, response_a=resp_a, response_b=resp_b
        )}],
    )
    text = msg.content[0].text.strip().upper() if msg.content else ""
    if "TIE" in text:
        return "TIE"
    elif text.startswith("A"):
        return "A"
    elif text.startswith("B"):
        return "B"
    return "TIE"

print("Judge ready. Run next cell to start evaluation.")

Judge ready. Run next cell to start evaluation.


In [14]:
judge_results_path = "./comparison_judge_results.json"

if os.path.exists(judge_results_path):
    with open(judge_results_path) as f:
        judge_results = json.load(f)
    print(f"Loaded {len(judge_results)} cached judge results")
else:
    dpo_gens = all_generations["dpo-seed42"]
    grpo_gens = all_generations["grpo-seed42"]

    judge_results = []
    for i in range(len(dpo_gens)):
        prompt = dpo_gens[i]["prompt"]
        dpo_resp = dpo_gens[i]["response"]
        grpo_resp = grpo_gens[i]["response"]

        fwd = judge_pair(prompt, dpo_resp, grpo_resp)
        rev = judge_pair(prompt, grpo_resp, dpo_resp)

        fwd_winner = {"A": "DPO", "B": "GRPO", "TIE": "TIE"}[fwd]
        rev_winner = {"A": "GRPO", "B": "DPO", "TIE": "TIE"}[rev]

        if fwd_winner == rev_winner:
            consistent = fwd_winner
        else:
            consistent = "INCONSISTENT"

        judge_results.append({
            "prompt_idx": i,
            "forward": fwd,
            "reverse": rev,
            "fwd_winner": fwd_winner,
            "rev_winner": rev_winner,
            "consistent": consistent,
        })

        if (i + 1) % 20 == 0:
            dpo_w = sum(1 for r in judge_results if r["consistent"] == "DPO")
            grpo_w = sum(1 for r in judge_results if r["consistent"] == "GRPO")
            print(f"  {i+1}/{len(dpo_gens)}: DPO={dpo_w}, GRPO={grpo_w}")

    with open(judge_results_path, "w") as f:
        json.dump(judge_results, f, indent=2)
    print(f"Saved {len(judge_results)} judge results")

  20/200: DPO=7, GRPO=3
  40/200: DPO=13, GRPO=8
  60/200: DPO=19, GRPO=11
  80/200: DPO=25, GRPO=17
  100/200: DPO=29, GRPO=22
  120/200: DPO=34, GRPO=27
  140/200: DPO=37, GRPO=33
  160/200: DPO=41, GRPO=36
  180/200: DPO=46, GRPO=43
  200/200: DPO=49, GRPO=49
Saved 200 judge results


In [15]:
n = len(judge_results)
dpo_wins = sum(1 for r in judge_results if r["consistent"] == "DPO")
grpo_wins = sum(1 for r in judge_results if r["consistent"] == "GRPO")
ties = sum(1 for r in judge_results if r["consistent"] == "TIE")
inconsistent = sum(1 for r in judge_results if r["consistent"] == "INCONSISTENT")

print("=== LLM Judge: DPO-seed42 vs GRPO-seed42 ===")
print(f"Total comparisons: {n}")
print(f"DPO wins:      {dpo_wins:3d} ({100*dpo_wins/n:.1f}%)")
print(f"GRPO wins:     {grpo_wins:3d} ({100*grpo_wins/n:.1f}%)")
print(f"Ties:          {ties:3d} ({100*ties/n:.1f}%)")
print(f"Inconsistent:  {inconsistent:3d} ({100*inconsistent/n:.1f}%)")

fwd_a = sum(1 for r in judge_results if r["forward"] == "A")
fwd_b = sum(1 for r in judge_results if r["forward"] == "B")
rev_a = sum(1 for r in judge_results if r["reverse"] == "A")
rev_b = sum(1 for r in judge_results if r["reverse"] == "B")
position_bias = (fwd_a + rev_a) / max(fwd_a + rev_a + fwd_b + rev_b, 1) - 0.5
print(f"\nPosition bias (toward A): {position_bias:+.3f}")
print(f"Forward: A={fwd_a}, B={fwd_b}, TIE={n-fwd_a-fwd_b}")
print(f"Reverse: A={rev_a}, B={rev_b}, TIE={n-rev_a-rev_b}")

=== LLM Judge: DPO-seed42 vs GRPO-seed42 ===
Total comparisons: 200
DPO wins:       49 (24.5%)
GRPO wins:      49 (24.5%)
Ties:           64 (32.0%)
Inconsistent:   38 (19.0%)

Position bias (toward A): +0.041
Forward: A=66, B=56, TIE=78
Reverse: A=66, B=56, TIE=78


## 5. Сводная таблица

In [16]:
def aggregate_by_method(keys, rm_scores, length_stats):
    rewards = [np.mean(rm_scores[k]) for k in keys]
    lengths = [length_stats[k]["mean_length"] for k in keys]
    trigrams = [length_stats[k]["unique_trigram_ratio"] for k in keys]
    return {
        "reward_mean": np.mean(rewards),
        "reward_std": np.std(rewards) if len(rewards) > 1 else 0,
        "length_mean": np.mean(lengths),
        "length_std": np.std(lengths) if len(lengths) > 1 else 0,
        "trigram_mean": np.mean(trigrams),
        "n_seeds": len(keys),
    }

sft_agg = aggregate_by_method(["sft-seed42"], rm_scores, length_stats)
dpo_keys = [f"dpo-seed{s}" for s in dpo_seeds if f"dpo-seed{s}" in rm_scores]
grpo_keys = [f"grpo-seed{s}" for s in grpo_seeds if f"grpo-seed{s}" in rm_scores]
dpo_agg = aggregate_by_method(dpo_keys, rm_scores, length_stats)
grpo_agg = aggregate_by_method(grpo_keys, rm_scores, length_stats)

print(f"{'Method':10s} {'Seeds':>5s} {'RM Reward':>14s} {'Resp Length':>14s} {'Trigram Uniq':>12s}")
print("-" * 60)

for name, agg in [("SFT", sft_agg), ("DPO", dpo_agg), ("GRPO", grpo_agg)]:
    reward_str = f"{agg['reward_mean']:+.4f}"
    if agg['reward_std'] > 0:
        reward_str += f" ± {agg['reward_std']:.4f}"
    length_str = f"{agg['length_mean']:.1f}"
    if agg['length_std'] > 0:
        length_str += f" ± {agg['length_std']:.1f}"
    print(f"{name:10s} {agg['n_seeds']:5d} {reward_str:>14s} {length_str:>14s} {agg['trigram_mean']:12.4f}")

print(f"\nNote: GRPO has {grpo_agg['n_seeds']} seed(s). Add seeds 43, 44 for full comparison.")

Method     Seeds      RM Reward    Resp Length Trigram Uniq
------------------------------------------------------------
SFT            1        +0.4174           52.0       0.8964
DPO            3 +0.5404 ± 0.0231     77.5 ± 0.8       0.8781
GRPO           1        +0.4815           78.4       0.8839

Note: GRPO has 1 seed(s). Add seeds 43, 44 for full comparison.


## 6. Per-seed breakdown

In [17]:
print(f"{'Model':20s} {'RM Reward':>12s} {'Resp Length':>12s} {'Trigram Uniq':>12s}")
print("-" * 60)

for key in ["sft-seed42"] + [f"dpo-seed{s}" for s in dpo_seeds] + [f"grpo-seed{s}" for s in grpo_seeds]:
    if key not in rm_scores:
        continue
    rm_mean = np.mean(rm_scores[key])
    stats = length_stats[key]
    print(f"{key:20s} {rm_mean:+12.4f} {stats['mean_length']:12.1f} {stats['unique_trigram_ratio']:12.4f}")

Model                   RM Reward  Resp Length Trigram Uniq
------------------------------------------------------------
sft-seed42                +0.4174         52.0       0.8964
dpo-seed42                +0.5383         76.7       0.8801
dpo-seed43                +0.5696         77.4       0.8802
dpo-seed44                +0.5131         78.5       0.8739
grpo-seed42               +0.4815         78.4       0.8839


## 7. Сохранение результатов

In [18]:
comparison_results = {
    "rm_scores_per_model": {k: {"mean": float(np.mean(v)), "std": float(np.std(v))} for k, v in rm_scores.items()},
    "length_stats": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in length_stats.items()},
    "aggregated": {
        "SFT": {k: float(v) if isinstance(v, (np.floating, float)) else v for k, v in sft_agg.items()},
        "DPO": {k: float(v) if isinstance(v, (np.floating, float)) else v for k, v in dpo_agg.items()},
        "GRPO": {k: float(v) if isinstance(v, (np.floating, float)) else v for k, v in grpo_agg.items()},
    },
    "judge": {
        "dpo_wins": dpo_wins,
        "grpo_wins": grpo_wins,
        "ties": ties,
        "inconsistent": inconsistent,
        "total": n,
        "position_bias": float(position_bias),
    },
    "dpo_seeds": dpo_seeds,
    "grpo_seeds": grpo_seeds,
}

with open("comparison_results.json", "w") as f:
    json.dump(comparison_results, f, indent=2, ensure_ascii=False)

print("Saved comparison_results.json")
print(json.dumps(comparison_results["aggregated"], indent=2))
print(json.dumps(comparison_results["judge"], indent=2))

Saved comparison_results.json
{
  "SFT": {
    "reward_mean": 0.41742706298828125,
    "reward_std": 0,
    "length_mean": 51.98,
    "length_std": 0,
    "trigram_mean": 0.8964107676969093,
    "n_seeds": 1
  },
  "DPO": {
    "reward_mean": 0.5403522904713948,
    "reward_std": 0.023108019315386967,
    "length_mean": 77.51666666666667,
    "length_std": 0.7593015796696928,
    "trigram_mean": 0.878067456481336,
    "n_seeds": 3
  },
  "GRPO": {
    "reward_mean": 0.48147491455078123,
    "reward_std": 0,
    "length_mean": 78.425,
    "length_std": 0,
    "trigram_mean": 0.8838545407064047,
    "n_seeds": 1
  }
}
{
  "dpo_wins": 49,
  "grpo_wins": 49,
  "ties": 64,
  "inconsistent": 38,
  "total": 200,
  "position_bias": 0.040983606557377095
}


# 8. Выводы

1. Оба алгоритма обходят SFT бейзлайн значительно по средней награде.
2. Оба алгоритма повышает длину ответа и немного снижают разнообразие.
3. DPO набирает более высокий RM reward, но независимый LLM-judge показывает ничью, что может говорить о том, что офлайн-метод лучше оптимизирует метрику, но не фактическое качество. Такое явление может быть связано с тем, что для DPO RM reward является таргетом.
4. GRPO эффективнее использует reward сигнал: достигает того же уровня качества по критику при меньшем RM score, то есть меньше переоптимизирует под конкретную RM.
5. Position bias судьи минимален, оценка методологически надёжна.